# 📔 Long-Term Episodic Memory

Episodic memory stores **time-stamped experiences** — what task was attempted and how it went —
so an agent can reference prior work instead of starting from zero. Like semantic memory
(`03_long_term_semantic_memory_sqlite.ipynb`), this is implemented with LangGraph's cross-thread `Store`;
the difference is entirely in *how it's written and read*: append-only, ordered by recency, and pruned
rather than updated in place.

## Learning Objectives
In this notebook, you will learn:
1. **Append-only episodes** - one fresh key per completed task
2. **Recency preload** - put the most recent episodes in the system prompt
3. **Recall in a new conversation** - reference earlier work from a different conversation
4. **Pruning** - keep only the most recent N episodes per user

## Prerequisites
- `03_long_term_semantic_memory_sqlite.ipynb` (memory tools that use `ToolRuntime`)
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

---
## 📦 Step 1: Environment Setup

The setup cell loads the API keys from the project-root `.env` and silences library deprecation warnings.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: completes tasks and recalls past ones
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🗒️ Part 1: Episodic Tools — Append-Only, Timestamp-Keyed

Unlike semantic memory's stable `key` (so re-saving updates the same item), each episode gets a
**fresh** key — a UUID — because two episodes are never the same fact restated, they're two
distinct events. `store.search` sorted by our own `created_at` field gives us "most recent N,"
the standard episodic read pattern.

Like the semantic tools, these tools reach the store and the trusted `user_id` through `ToolRuntime`.

In [ ]:
# ============================================================================
# PART 1: AN EPISODIC STORE AND TWO TOOLS
# ============================================================================
import json
import os
import sqlite3
import uuid
from dataclasses import dataclass
from datetime import datetime, timezone

from langchain.tools import ToolRuntime, tool
from langgraph.store.sqlite import SqliteStore

DB_PATH = "episodic_memory.db"
RESET_DB = True  # False keeps memories from earlier runs
if RESET_DB and os.path.exists(DB_PATH):
    os.remove(DB_PATH)
conn = sqlite3.connect(DB_PATH, check_same_thread=False, isolation_level=None)
episodic_store = SqliteStore(conn)
episodic_store.setup()


@dataclass
class Context:
    user_id: str  # trusted: set by your server from the login, never by the model


def episodes_newest_first(store, user_id: str) -> list:
    """All of a user's episodes, newest first, sorted on our own created_at field."""
    # search() returns 10 items by default -- ask for more, then sort ourselves
    # ponytail: one page of 100; pruning (Part 4) keeps a user well below it
    items = store.search(("episodic", user_id), limit=100)
    return sorted(items, key=lambda it: it.value["created_at"], reverse=True)


@tool
def save_episode(task: str, outcome: str, runtime: ToolRuntime[Context], feedback: str = "") -> str:
    """Record a completed task and its outcome as a new episode (never overwrites past episodes)."""
    runtime.store.put(
        ("episodic", runtime.context.user_id),
        str(uuid.uuid4()),  # a fresh key per event: append-only
        {
            "task": task,
            "outcome": outcome,
            "feedback": feedback,
            "created_at": datetime.now(timezone.utc).isoformat(),
        },
    )
    return "Episode recorded."


@tool
def recall_recent_episodes(runtime: ToolRuntime[Context], limit: int = 3) -> str:
    """Return the most recent episodes for this user."""
    recent = [it.value for it in episodes_newest_first(runtime.store, runtime.context.user_id)[:limit]]
    return json.dumps(recent, indent=2) if recent else "No past episodes."


episodic_tools = [save_episode, recall_recent_episodes]
print("✅ Episodic store and tools ready")

---
## 📥 Part 2: The Agent — Preload Recent Episodes

The node puts the three most recent episodes into the system prompt, so a new conversation starts
with a short record of what was already done. The graph has no checkpointer: every call is a
brand-new conversation.

In [ ]:
# ============================================================================
# PART 2: THE MEMORY-AWARE AGENT GRAPH
# ============================================================================
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.runtime import Runtime

llm_with_tools = llm.bind_tools(episodic_tools)

SYSTEM_TEMPLATE = """You are a helpful assistant with persistent episodic memory.

Recent past interactions with this user:
{episodes}

After you complete a substantive task, call `save_episode` with a short task/outcome summary
so future conversations can reference what was already tried.
"""


def agent_node(state: MessagesState, runtime: Runtime[Context]) -> dict:
    recent = episodes_newest_first(runtime.store, runtime.context.user_id)[:3]
    episodes_block = "\n".join(f"- {it.value['task']} -> {it.value['outcome']}" for it in recent) or "(none)"

    system = SystemMessage(SYSTEM_TEMPLATE.format(episodes=episodes_block))
    response = llm_with_tools.invoke([system] + state["messages"])
    return {"messages": [response]}


builder = StateGraph(MessagesState, context_schema=Context)
builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode(episodic_tools))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition, ["tools", END])
builder.add_edge("tools", "agent")

episodic_agent = builder.compile(store=episodic_store)


def chat(text: str, user_id: str, max_chars: int = 600) -> None:
    """One brand-new conversation for `user_id`: prints the tool calls and the (shortened) answer."""
    result = episodic_agent.invoke({"messages": [HumanMessage(text)]}, context=Context(user_id=user_id))
    calls = [tc["name"] for m in result["messages"] for tc in getattr(m, "tool_calls", [])]
    print(f"[{user_id}] USER: {text}")
    print("  tool calls:", calls or "none")
    print("  AGENT:", result["messages"][-1].content[:max_chars], "\n")


print("✅ Episodic-memory agent compiled")

---
## 🧵 Part 3: Recall in a New Conversation

### 3.1 A Task Gets Logged as an Episode

The user's message ends with "save this as an episode once you're done". That request is a **teaching
shortcut**: it makes the save happen on cue, so the demo repeats reliably. Real users don't ask for this.
Left to itself, gpt-4o-mini rarely calls `save_episode` — in test runs of the capstone agent it never did
(0 of 6) — so `06_combined_memory_agent_sqlite.ipynb` writes episodes in a dedicated graph node after each
answer instead of leaving the decision to the model.

In [ ]:
# ============================================================================
# PART 3.1: COMPLETE A TASK (AND ASK FOR THE SAVE -- A TEACHING SHORTCUT)
# ============================================================================
USER_ID = "user-alice"
chat(
    "Help me outline a 3-step plan to migrate a Spark job from Scala to PySpark, "
    "then save this as an episode once you're done.",
    USER_ID,
)
for it in episodes_newest_first(episodic_store, USER_ID):
    print("stored episode:", it.value["task"], "->", it.value["outcome"])

### 3.2 A Later, New Conversation References the Past Episode

In [ ]:
# ============================================================================
# PART 3.2: NEW CONVERSATION, EPISODE RECALLED
# ============================================================================
chat("What did you help me with last time?", USER_ID)

### 3.3 Semantic vs. Episodic — Side by Side

Both notebooks use the exact same `Store` mechanism (`SqliteStore`, namespaced by `user_id`, reached
through `ToolRuntime`) — the difference is entirely in the write/read pattern, not the storage primitive:

| | Semantic (`03_long_term_semantic_memory_sqlite.ipynb`) | Episodic (this notebook) |
|---|---|---|
| Key | Stable (`"name"`, `"preferred_language"`) | Fresh UUID per event |
| Write behavior | Update-in-place | Append-only |
| Read pattern | Load everything | Load most-recent-N by `created_at` |

---
## ✂️ Part 4: Pruning

Episodic memory grows with every completed task — unlike semantic memory, there's no natural
cap on item count. A retention policy keeps only the most recent `N` episodes per user.

The padding episodes below are written directly to `episodic_store`. Code outside a graph run has no
`runtime` to inject into a tool, so setup and maintenance code — like this pruning job — works on the
store object itself.

In [ ]:
# ============================================================================
# PART 4: KEEP ONLY THE MOST RECENT N EPISODES
# ============================================================================
def prune_old_episodes(user_id: str, keep_last: int = 20) -> int:
    to_delete = episodes_newest_first(episodic_store, user_id)[keep_last:]
    for it in to_delete:
        episodic_store.delete(("episodic", user_id), it.key)
    return len(to_delete)


# Three more episodes, so there is something to prune
for i in range(3):
    episodic_store.put(
        ("episodic", USER_ID),
        str(uuid.uuid4()),
        {
            "task": f"Demo task {i}",
            "outcome": f"Demo outcome {i}",
            "feedback": "",
            "created_at": datetime.now(timezone.utc).isoformat(),
        },
    )

before = len(episodes_newest_first(episodic_store, USER_ID))
deleted = prune_old_episodes(USER_ID, keep_last=2)
kept = [it.value["task"] for it in episodes_newest_first(episodic_store, USER_ID)]
print(f"episodes before: {before}, pruned: {deleted}, kept (newest first): {kept}")

---
## 📝 Summary

In this notebook, we learned:

### 1. Append-only episodes
- Episodic memory reuses the same `Store` primitive as semantic memory, with a fresh key per event

### 2. Recency preload
- The node preloads the most recent few episodes, sorted on our own `created_at` field

### 3. Recall in a new conversation
- A brand-new conversation can say what was done last time, because the episode lives in the store

### 4. Pruning
- Preload the most recent few; prune or summarize the rest — this layer has a growth problem semantic memory
  doesn't

### 5. Gotchas
- **Append-only means unbounded growth is the default, not an edge case.** Every completed task adds an item;
  without a pruning policy like Part 4's (or summarization into a compacted form), a long-lived user's episode
  count grows without limit
- **Models rarely save episodes on their own.** Part 3.1 asked for the save explicitly — a teaching shortcut.
  Production agents write episodes in code after each run (see `06_combined_memory_agent_sqlite.ipynb`) or in a
  background job
- **"Recent N" preloading can miss the relevant episode.** A most-recent-3 preload won't surface something from
  months ago. Retrieve by meaning instead: `search(..., query=...)` on an embedding-indexed store, or the
  vector-store episodes of [`01_episodic_with_semantic_memory.ipynb`](04a_episodic_with_semantic_memory.ipynb) (module 04)
- **Sorting is your responsibility, and `search()` returns 10 items by default.** This notebook asks for
  `limit=100` and sorts on its own `created_at` field; trusting the backend's order or the default page size
  silently loses episodes
- **Don't duplicate the conversation transcript here.** `save_episode` should record a *summary* ("outlined a
  3-step migration plan"), not the full exchange — the full exchange is short-term memory's job (module 02's
  `01_memory_and_conversational_agent.ipynb`)
- **Episodes can embed PII** in `task`/`outcome` text just as easily as semantic facts. Trusted ids, access
  control and SQLite concurrency apply to every layer: see [`memory_layers_guide.md`](../assets/memory_layers_guide.md)

### Next Steps
- **`05_long_term_procedural_memory_sqlite.ipynb`**: versioned instructions that change behaviour, behind human approval
- **[`01_episodic_with_semantic_memory.ipynb`](04a_episodic_with_semantic_memory.ipynb)** (module 04): episodes recalled by similarity search instead of recency